## 4. 自然语言处理
这一章主要介绍大语言模型在自然语言处理中的应用。自然语言处理大体上可分为自然语言理解（Natural Language Understanding，以下简称NLU）和自然语言生成（Natural Language Generation，以下简称NLG）两大类，仅编码器模型非常适合处理NLU，而编码器-解码器模型和仅解码器模型则更适合处理NLG。这一章大体上也是按大模型处理任务的类型来组织的，先NLU，再NLG。

### 4.1 自然语言理解任务
transformers中的自然语言理解任务主要包括文本分类、词元分类和问答等。

#### 4.1.1 文本分类
文本分类（Text Classification）是根据文本内容的语义含义，将它们逐一划分到预定义的一组类别中。简单来说就是给文本打标签，标签需要预先定义好。文本分类最典型的应用之一就是情感分析（Sentiment Analysis），它可以根据用户的留言、问答或评价等内容，分析其中的情感色彩是正面的、中性的还是负面的。除了情感分析以外，文本分类还可用于投诉分类、内容审核、邮件分类等领域。
transformers库中文本分类的任务名称是text-classification，但需要注意transformers库的默认模型只能做情感分析。如果需要做其他类型的文本分类，需要使用相应的数据集对模型进行微调。具体代码示例如下：

In [ ]:
from transformers import pipeline

classifier = pipeline(task="sentiment-analysis")
result = classifier("I like this book.")
print(result)

#### 4.1.2 词元分类
词元分类（Token Classification）与文本分类类似，只不过分类的对象由整段文本变成细粒度的词元。词元分类最典型的应用场景就是命名实体识别（Named Entity Recognition，NER），目的是从整段文本中识别组织机构、人名地名、时间数字、位置坐标等各类实体信息。除了NER以外，词元分类还可以应用在关键词提取、拼写检查、词性标注等场景。但同样的，transformers库提供的默认模型只能做NER，应用于其它场景时需要进行微调：

In [ ]:
from transformers import pipeline

classifier = pipeline(task="ner")
preds = classifier("I'm Tian Xuesong from Beijing China.")
print(preds)

#### 4.1.3 问答
问答（Question Answering，QA）任务可分为抽取式问答（Extractive QA）和生成式问答（Generative QA）两种。transformers框架中的问答任务是抽取式问答，使用的任务名称为question-answering。transformers处理问答任务的代码与其它任务类似，不过在推理时需要通过question和context参数同时提供问题和上下文：

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering")
result = qa(question="What's the capital of USA?", context='''
        The United States, with its capital in Washington, D.C., 
        is a federal republic known for diversity, 
        innovation, and global influence.''')
print(result)

### 4.2 自然语言生成任务
NLG主要包括翻译、摘要和文本生成等几种，转换器模型在这几种任务上都展现了巨大的进步。NLG主要由编码器-解码器和仅解码器两种形态的模型实现，但借助transformers库的流水线机制，它们在代码实现上与NLU任务并没有明显不同。

#### 4.2.1 翻译
transformers中翻译任务的名称为translation，但直接使用这个名称加载模型可能会报错。这是因为t5-base模型需要预先指定要翻译的语言对，可通过translation_XX_YY的形式提供给pipeline函数，其中的XX和YY就是语言对的两字母语言代码。

In [ ]:
from transformers import pipeline

text = "I love this book."
translator = pipeline(task="translation_en_to_de")
result = translator(text)
print(result)

目前，transformers默认模型只支持从英语到法语、德语和罗马尼亚语的翻译任务，即只有en_fr、en_de和en_ro是合法的，其它语言对组合都会报错。但一些规模较大的模型已经可以支持任意语言对之间的直接翻译了。比如Facebook发布的M2M-100模型，可以支持多达100种语言的相互翻译：

In [ ]:
from transformers import pipeline

text = "我喜欢这本书."
translator = pipeline(task="translation", model="facebook/m2m100_418M", 
src_lang="zh", tgt_lang="de")
result = translator(text)
print(result)

#### 4.2.2 摘要
摘要（Summarization）是将较长文本或文本集合浓缩为简短文本的任务，浓缩后的文本应尽可能保留原文本中的关键信息。transformers摘要任务的名称为summarization，实现过程与前述任务类似：

In [ ]:
from transformers import pipeline

summarizer = pipeline(task="summarization", min_length=50, max_length=80)
sum =summarizer('''
Generative AI, sometimes called gen AI, is artificial intelligence (AI) that can create original content—such as text, images, video, audio or software code—in response to a user's prompt or request.
Generative AI relies on sophisticated machine learning models called deep learning models—algorithms that simulate the learning and decision-making processes of the human brain. 
These models work by identifying and encoding the patterns and relationships in huge amounts of data, and then using that information to understand users' natural language requests or questions and respond with relevant new content.
AI has been a hot technology topic for the past decade, but generative AI, and specifically the arrival of ChatGPT in 2022, has thrust AI into worldwide headlines and launched an unprecedented surge of AI innovation and adoption. 
Generative AI offers enormous productivity benefits for individuals and organizations, and while it also presents very real challenges and risks, businesses are forging ahead, exploring how the technology can improve their internal workflows and enrich their products and services. 
According to research by the management consulting firm McKinsey, one third of organizations are already using generative AI regularly in at least one business function. 
Industry analyst Gartner projects more than 80% of organizations will have deployed generative AI applications or used generative AI application programming interfaces (APIs) by 2026.
'''
)

print(sum)

#### 4.2.3 文本生成
文本生成（Text Generation）应该说是转换器模型最成功的应用之一，也是大语言模型生成力与创造力的重要体现。transformers库使用的默认模型为GPT2。文本生成的任务名称为text-generation，所以通过pipeline也可像其它任务一样轻松实现文本生成：

In [ ]:
from transformers import pipeline

prompt = "The book is super good."
generator = pipeline(task="text-generation")
text = generator(prompt)
print(text)

这一小节还介绍了许多生成时使用的参数，具体请参考书中介绍。

#### 4.2.4 填充掩码
填充掩码（Fill Mask）也可以称为填空，它是根据文本序列上下文预测文本中被掩码（通常是标识为<mask>）的部分。填充掩码并不能算是NLG任务，之所以将其放在这一部分，是因为它与文本生成一样，属于语言建模（Language Modeling）的一种类型。

In [ ]:
from transformers import pipeline

text = "The book is <mask> for generative ai."
fill_mask = pipeline(task="fill-mask")
preds = fill_mask(text, top_k=2)
print(preds)

### 4.3  解构流水线
这一节介绍了transformers库流水线（Pipeline）的内部实现，这在需要对模型进行微调时尤其重要。流水线之所以称为流水线是因为它将处理过程拆分为多个组件，并将它们按顺序组装起来，就像流水线一样处理任务。下图展示这些组件之间的关系：

![流水线](./images/pipeline.png)

transformers库的模型类大致可分为三个层次的抽象，它们是框架抽象层、模型抽象层和任务抽象层。框架抽象层对应着不同的底层框架，模型抽象层则对应着具体的大语言模型，而任务抽象层是对不同任务的适配。图4-3以BERT模型的实现类为例，展示了这些抽象层之间的关系：

![模型抽象](./images/model.png)

### 4.4  本章小结
本章主要介绍了转换器模型在自然语言处理任务上的应用，同时对transformers的流水线机制做了深度解析。
自然语言处理任务可分为自然语言理解和自然语言生成两个大的类别，它们实际上就是自然语言领域的判别式任务和生成式任务。自然语言理解任务包括文本分类、词元分类、问答和填空等，而自然语言生成则包括翻译、摘要和文本生成等。在执行自然语言理解任务时，transformers通常会采用类似BERT这样的仅编码器模型，并通过添加特定的任务头来适配不同的判别任务。而在执行自然语言生成任务时，transformers则会采用T5、BART这样的编码器-解码器模型，或是GPT这种仅解码器模型。这些模型在生成文本时可以通过一些参数来控制生成文本的质量，包括temperature、top_p、top_k以及nub_beams等。
transformers流水线机制将处理人工智能所需组件封装了起来，包括预处理、前向传播和后处理等三个主要步骤。针对不同业务数据，流水线每个步骤所使用的组件可能有所不同，但整体处理流程大同小异。对于自然语言处理任务来说，预处理主要是使用分词器对文本进行分词，前向传播则是利用模型做推理的过程，而后处理是将模型推理结果转换为用户友好的形式。transformers为不同方便不同组件的加载，分别为它们提供了相应的自动加载类，可通用模型名称、路径等形式加载组件。这为后续对不同模型进行训练和微调，或是定制流水线处理流程提供了便利。